In [ ]:
!pip -q install pyarrow

from google.colab import drive
from pathlib import Path
from IPython.display import display

import gc
import json
import unicodedata

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

drive.mount("/content/drive")

# 한 번에 처리할 행 수
BATCH_SIZE = 20_000

# 같은업태 변수가 추가된 40개월을 처리
months = pd.date_range(
    "2023-01-01",
    "2026-01-31",
    freq="MS",
)

In [ ]:
def resolve_korean_path(path):
    target = Path(path)
    current = Path(target.anchor)

    for part in target.parts[1:]:
        direct = current / part

        if direct.exists():
            current = direct
            continue

        matches = [
            item for item in current.iterdir()
            if unicodedata.normalize("NFC", item.name)
            == unicodedata.normalize("NFC", part)
        ]

        if len(matches) != 1:
            raise FileNotFoundError(
                f"경로 확인 필요: {current / part}"
            )

        current = matches[0]

    return current


BASE = resolve_korean_path(
    "/content/drive/MyDrive/KB_FinalProject_Team6/"
    "완전쓰는거_ver2"
)

MASTER_PATH = resolve_korean_path(
    str(BASE / "인허가_master.csv")
)

SOURCE_DIR = resolve_korean_path(
    str(
        BASE
        / "인허가_파생지표_202301_202604"
        / "같은업태추가"
        / "snapshots"
    )
)

RESULT_DIR = BASE / "인허가_최종데이터_한글컬럼"
VIZ_DIR = RESULT_DIR / "시각화용"
MODEL_DIR = RESULT_DIR / "모델링용"

VIZ_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

source_files = [
    SOURCE_DIR / f"snapshot_{t:%Y%m}.parquet"
    for t in months
]

missing = [
    str(path)
    for path in source_files
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        "누락된 월별 파일:\n" + "\n".join(missing)
    )

print("원본:", MASTER_PATH)
print("월별 입력:", SOURCE_DIR)
print("결과 저장:", RESULT_DIR)
print("처리할 월:", len(source_files))

In [ ]:
for encoding in ["utf-8-sig", "cp949"]:
    try:
        header = pd.read_csv(
            MASTER_PATH,
            encoding=encoding,
            nrows=0,
        )
        break
    except UnicodeDecodeError:
        continue
else:
    raise ValueError("원본 CSV 인코딩을 확인하세요.")

industry_cols = [
    col for col in header.columns
    if col.startswith("업종__")
]

if not industry_cols:
    raise ValueError(
        "원본에 업종__로 시작하는 컬럼이 없습니다."
    )

# 0·1 컬럼은 작은 정수형으로 읽어 메모리 절약
read_types = {
    "점포이력ID": "string",
    **{col: "Int8" for col in industry_cols},
}

industry = pd.read_csv(
    MASTER_PATH,
    encoding=encoding,
    usecols=["점포이력ID"] + industry_cols,
    dtype=read_types,
)

if (
    industry["점포이력ID"].isna().any()
    or industry["점포이력ID"].duplicated().any()
):
    raise ValueError("원본 점포이력ID 결측·중복을 확인하세요.")

for col in industry_cols:
    if (
        industry[col].isna().any()
        or not industry[col].isin([0, 1]).all()
    ):
        raise ValueError(
            f"{col}: 결측 또는 0·1 이외 값이 있습니다."
        )

industry = industry.set_index("점포이력ID")
industry.index.name = "점포번호"

# 검증 완료 후 일반 int8로 변환
industry = industry.astype("int8")

print(f"점포 수: {len(industry):,}")
print(f"업종별 컬럼 수: {len(industry_cols)}")

memory_mb = (
    industry.memory_usage(deep=True).sum()
    / 1024**2
)

print(f"업종 연결표 메모리: {memory_mb:,.1f} MB")

display(industry.head(3))


In [ ]:
# 기존 월별 파일의 컬럼명 → 최종 한글명
column_map = {
    "base_ym": "기준월",
    "점포이력ID": "점포번호",
    "open_s": "개업일",
    "lon": "경도",
    "lat": "위도",
    "coord_valid": "좌표유효여부",

    "age_m": "영업기간_개월",
    "age_band": "영업기간_구간",
    "is_new": "개업1년미만",
    "biz_type": "점포업종",
    "is_franchise": "프랜차이즈여부",

    "dong_age_med": "동네점포_영업기간중앙값_개월",
    "closed_life_avg": "주변동종_폐업전평균영업기간_개월",
    "surv_6m": "주변동종_신규점포_6개월생존율",
    "fr_share": "주변동종_프랜차이즈비율",
    "close_rate_6m": "주변동종_6개월폐업률",
    "open_rate_6m": "주변동종_6개월개업률",
    "net_6m": "주변동종_6개월점포증감",
    "biz_entropy": "동네_업종다양성",

    "same_300m": "300미터내_동종점포수",
    "close_100m_6m": "100미터내_6개월폐업수",

    "n_hk": "동네_동종점포수",
    "n_gk": "구군_동종점포수",
    "n_start_hk": "동네同종_6개월전점포수".replace("同", "동"),
    "n_open_hk": "동네동종_6개월개업수",
    "n_close_hk": "동네동종_6개월폐업수",
    "n_dong_age": "동네_영업기간계산점포수",
    "n_closed_life_h": "동네동종_1년폐업수",
    "n_cohort_h": "동네동종_생존율계산점포수",

    "label_close_6m": "앞으로6개월내_폐업여부",
}

# 既存 한글 컬럼은 이름 그대로 사용
for col in [
    "구군",
    "행정동코드",
    "행정동명",
    "법정동",
    "업종명",
    "업태구분명",
    "동네_같은업태점포수",
    "구군_같은업태점포수",
    "300미터내_같은업태점포수",
]:
    column_map[col] = col

derived_features = [
    "영업기간_개월",
    "영업기간_구간",
    "개업1년미만",
    "점포업종",
    "프랜차이즈여부",

    "동네점포_영업기간중앙값_개월",
    "주변동종_폐업전평균영업기간_개월",
    "주변동종_신규점포_6개월생존율",
    "주변동종_프랜차이즈비율",
    "주변동종_6개월폐업률",
    "주변동종_6개월개업률",
    "주변동종_6개월점포증감",
    "동네_업종다양성",

    "300미터내_동종점포수",
    "100미터내_6개월폐업수",

    "동네_같은업태점포수",
    "구군_같은업태점포수",
    "300미터내_같은업태점포수",
]

key_cols = ["기준월", "점포번호"]
target_col = "앞으로6개월내_폐업여부"

feature_cols = derived_features + industry_cols
model_cols = key_cols + feature_cols + [target_col]

viz_candidates = [
    "기준월",
    "점포번호",
    "구군",
    "행정동코드",
    "행정동名".replace("名", "명"),
    "법정동",
    "업종명",
    "업태구분명",
    "개업일",
    "경도",
    "위도",
    "좌표유효여부",

    "동네_동종점포수",
    "구군_동종점포수",
    "동네동종_6개월전점포수",
    "동네동종_6개월개업수",
    "동네동종_6개월폐업수",
    "동네_영업기간계산점포수",
    "동네동종_1년폐업수",
    "동네동종_생존율계산점포수",

    *derived_features,
    *industry_cols,
    target_col,
]

# 실제 입력 파일에 존재하는 시각화 컬럼만 선정
first_columns = pq.read_schema(source_files[0]).names

available_korean = {
    column_map.get(col, col)
    for col in first_columns
} | set(industry_cols)

viz_cols = list(dict.fromkeys(
    col for col in viz_candidates
    if col in available_korean
))

# 모든 월에 필요한 컬럼이 있는지 먼저 검사
required_korean = set(
    key_cols + derived_features + [target_col, "업태구분명"]
)

for path in source_files:
    names = pq.read_schema(path).names

    renamed = {
        column_map.get(col, col)
        for col in names
    }

    missing = required_korean - renamed

    if missing:
        raise ValueError(
            f"{path.name} 필수 컬럼 누락: {sorted(missing)}"
        )

    # 시각화용 선정 컬럼도 모든 월에서 동일하게 사용
    missing_viz = (
        set(viz_cols) - set(industry_cols)
    ) - renamed

    if missing_viz:
        raise ValueError(
            f"{path.name} 시각화 컬럼 누락: {sorted(missing_viz)}"
        )

# 필요한 원본 컬럼만 읽도록 설정
needed_output = set(viz_cols + model_cols) - set(industry_cols)

source_columns = [
    col for col in first_columns
    if column_map.get(col, col) in needed_output
    and col not in industry_cols
]

# 문자열 컬럼은 배치마다 동일한 자료형 사용
text_cols = {
    "기준월",
    "점포번호",
    "구군",
    "행정동코드",
    "행정동명",
    "법정동",
    "업종명",
    "업태구분명",
    "점포업종",
    "영업기간_구간",
}

with open(
    RESULT_DIR / "모델입력컬럼.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        {
            "모델입력": feature_cols,
            "범주형입력": [
                "영업기간_구간",
                "점포업종",
            ],
            "추적정보": key_cols,
            "정답": target_col,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

pd.DataFrame([
    {
        "기존컬럼명": col,
        "한글컬럼명": column_map.get(col, col),
    }
    for col in source_columns
]).to_csv(
    RESULT_DIR / "컬럼명변경표.csv",
    index=False,
    encoding="utf-8-sig",
)

print("파생지표:", len(derived_features))
print("업종별 0·1:", len(industry_cols))
print("모델 입력 후보:", len(feature_cols))
print("시각화용 컬럼:", len(viz_cols))

In [ ]:
def prepare_batch(batch):
    frame = batch.to_pandas()

    # 범주형을 문자열로 바꿔 배치별 범주 차이 방지
    for col in frame.select_dtypes(include="category").columns:
        frame[col] = frame[col].astype("string")

    frame = frame.rename(columns=column_map)

    for col in text_cols:
        if col in frame.columns:
            frame[col] = frame[col].astype("string")

    frame["개업일"] = pd.to_datetime(
        frame["개업일"],
        errors="raise",
    ) if "개업일" in frame else pd.NaT

    frame[target_col] = pd.to_numeric(
        frame[target_col],
        errors="raise",
    ).astype("Int8")

    if frame["점포번호"].isna().any():
        raise ValueError("점포번호 결측이 있습니다.")

    # 원본 업종 정보와 연결되는지 확인
    linked = frame["점포번호"].isin(industry.index)

    if not linked.all():
        raise ValueError(
            f"원본과 연결되지 않는 점포: {(~linked).sum():,}개"
        )

    # 큰 merge 대신 작은 배치에 업종별 배열만 붙이기
    flags = industry.reindex(frame["점포번호"])

    for col in industry_cols:
        frame[col] = flags[col].to_numpy(dtype="int8")

    del flags

    return frame


def write_part(frame, columns, writer, schema, temp_path):
    table = pa.Table.from_pandas(
        frame.loc[:, columns],
        preserve_index=False,
    ).replace_schema_metadata(None)

    if writer is None:
        schema = table.schema
        writer = pq.ParquetWriter(
            temp_path,
            schema,
            compression="snappy",
        )
    else:
        table = table.cast(schema)

    writer.write_table(table)

    del table
    return writer, schema


reports = []

for month_number, (t, source_path) in enumerate(
    zip(months, source_files),
    start=1,
):
    viz_path = VIZ_DIR / f"시각化用_{t:%Y%m}.parquet".replace(
        "化用", "화용"
    )
    model_path = MODEL_DIR / f"모델링용_{t:%Y%m}.parquet"

    viz_temp = viz_path.with_suffix(".parquet.part")
    model_temp = model_path.with_suffix(".parquet.part")

    viz_writer = None
    model_writer = None
    viz_schema = None
    model_schema = None

    row_count = 0
    label_count = 0

    parquet_file = pq.ParquetFile(source_path)
    expected_rows = parquet_file.metadata.num_rows

    try:
        for batch_number, batch in enumerate(
            parquet_file.iter_batches(
                batch_size=BATCH_SIZE,
                columns=source_columns,
                use_threads=False,
            ),
            start=1,
        ):
            frame = prepare_batch(batch)

            if not frame["기준월"].eq(t.strftime("%Y-%m")).all():
                raise ValueError(
                    f"{source_path.name}: 기준월이 일치하지 않습니다."
                )

            # 두 결과를 순서대로 기록해 임시 메모리 사용 최소화
            viz_writer, viz_schema = write_part(
                frame,
                viz_cols,
                viz_writer,
                viz_schema,
                viz_temp,
            )

            model_writer, model_schema = write_part(
                frame,
                model_cols,
                model_writer,
                model_schema,
                model_temp,
            )

            row_count += len(frame)
            label_count += int(
                frame[target_col].notna().sum()
            )

            del frame, batch
            gc.collect()

            print(
                f"[{month_number:02d}/40] {t:%Y-%m} "
                f"| 배치 {batch_number} "
                f"| {row_count:,}/{expected_rows:,}행",
                flush=True,
            )

    finally:
        if viz_writer is not None:
            viz_writer.close()

        if model_writer is not None:
            model_writer.close()

    if row_count != expected_rows:
        raise ValueError(
            f"{t:%Y-%m}: 입력·출력 행 수가 다릅니다."
        )

    # 파일 행 수까지 확인한 다음 최종 이름으로 변경
    for temp in [viz_temp, model_temp]:
        saved_rows = pq.ParquetFile(temp).metadata.num_rows

        if saved_rows != expected_rows:
            raise ValueError(f"저장 행 수 불일치: {temp}")

    viz_temp.replace(viz_path)
    model_temp.replace(model_path)

    reports.append({
        "기준월": t.strftime("%Y-%m"),
        "행수": row_count,
        "정답있는행수": label_count,
        "시각화파일": str(viz_path),
        "모델링파일": str(model_path),
    })

    pd.DataFrame(reports).to_csv(
        RESULT_DIR / "처리결과요약.csv",
        index=False,
        encoding="utf-8-sig",
    )

    del parquet_file
    gc.collect()

    print(f"→ {t:%Y-%m} 저장 완료\n", flush=True)

print("40개월 처리 완료")
display(pd.DataFrame(reports))

In [ ]:
from google.colab import drive
from pathlib import Path
import unicodedata
import pyarrow.parquet as pq

drive.mount("/content/drive")

root = Path(
    "/content/drive/MyDrive/KB_FinalProject_Team6"
)

def normalize(text):
    return unicodedata.normalize("NFC", text)

# 한글 자모 분리 차이를 고려해 파일 검색
all_parquet = list(root.rglob("*.parquet"))

full_name = "모델링용_전체_202301_202604.parquet"

full_matches = [
    path for path in all_parquet
    if normalize(path.name) == full_name
]

if len(full_matches) == 1:
    model_path = full_matches[0]
    print("전체 모델링 파일을 찾았습니다.")

elif len(full_matches) > 1:
    raise ValueError(
        "전체 파일이 여러 개 있습니다:\n"
        + "\n".join(map(str, full_matches))
    )

else:
    monthly_matches = sorted([
        path for path in all_parquet
        if normalize(path.name).startswith("모델링용_")
        and normalize(path.name)[len("모델링용_"):-len(".parquet")].isdigit()
        and len(
            normalize(path.name)[len("모델링용_"):-len(".parquet")]
        ) == 6
    ], key=lambda path: normalize(path.name))

    if not monthly_matches:
        raise FileNotFoundError(
            "전체 파일과 월별 모델링 파일을 모두 찾지 못했습니다. "
            "드라이브의 실제 저장 위치를 확인해주세요."
        )

    model_path = monthly_matches[-1]

    print("전체 파일은 찾지 못했습니다.")
    print(f"월별 모델링 파일은 {len(monthly_matches)}개 발견됐습니다.")
    print("가장 최신 월 파일로 표본을 확인합니다.")

print("\n사용 파일:", model_path)

source = pq.ParquetFile(model_path)

print(f"파일 전체 행 수: {source.metadata.num_rows:,}")
print(f"컬럼 수: {len(source.schema_arrow.names):,}")

batch = next(
    source.iter_batches(
        batch_size=1_000,
        use_threads=False,
    ),
    None,
)

if batch is None:
    raise ValueError("파일에 데이터가 없습니다.")

df_model = batch.to_pandas()

display(df_model.head(20))
print("\n컬럼명:", df_model.columns.tolist())

In [ ]:
from google.colab import drive
from pathlib import Path
from IPython.display import display

import unicodedata
import gc
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

drive.mount("/content/drive")

ROOT = Path(
    "/content/drive/MyDrive/KB_FinalProject_Team6"
)

BATCH_SIZE = 20_000

# 2026년 1월 말 정보가 반영된 2월 1일 스냅샷까지 포함
months = pd.date_range(
    "2023-01-01",
    "2026-02-01",
    freq="MS",
)

def normalize(text):
    return unicodedata.normalize("NFC", text)

expected_names = [
    f"모델링용_{month:%Y%m}.parquet"
    for month in months
]

expected_set = set(expected_names)
found = {}

for path in ROOT.rglob("*.parquet"):
    name = normalize(path.name)

    if name in expected_set:
        found.setdefault(name, []).append(path)

missing = [
    name for name in expected_names
    if name not in found
]

if missing:
    raise FileNotFoundError(
        "누락된 월별 파일:\n" + "\n".join(missing)
    )

duplicates = {
    name: matches
    for name, matches in found.items()
    if len(matches) > 1
}

if duplicates:
    raise ValueError(
        "동일한 이름의 파일이 여러 개 있습니다:\n"
        + "\n".join(
            str(path)
            for matches in duplicates.values()
            for path in matches
        )
    )

paths = [found[name][0] for name in expected_names]

if len({path.parent for path in paths}) != 1:
    raise ValueError(
        "월별 파일이 서로 다른 폴더에 있습니다. "
        "입력 파일을 확인하세요."
    )

output_path = (
    paths[0].parent.parent
    / "모델링용_전체_202301_202602.parquet"
)

print(f"스냅샷: {months[0]:%Y-%m-%d} ~ {months[-1]:%Y-%m-%d}")
print(f"월별 파일: {len(paths)}개")
print("마지막 입력 정보 기준: 2026년 1월 31일까지")
print("저장 위치:", output_path)

In [ ]:
def normalize_schema(schema):
    # 월별 범주형 차이를 처리하기 위해 기본 자료형으로 통일
    return pa.schema([
        pa.field(
            field.name,
            (
                field.type.value_type
                if pa.types.is_dictionary(field.type)
                else field.type
            ),
            nullable=True,
        )
        for field in schema
    ])


# 데이터 대신 작은 스키마 정보만 먼저 읽기
schemas = [
    normalize_schema(pq.read_schema(path))
    for path in paths
]

columns = schemas[0].names

for path, schema in zip(paths, schemas):
    if set(schema.names) != set(columns):
        raise ValueError(
            f"컬럼 구성이 다릅니다: {path.name}"
        )

required = [
    "기준월",
    "점포번호",
    "앞으로6개월내_폐업여부",
]

missing_columns = [
    col for col in required
    if col not in columns
]

if missing_columns:
    raise ValueError(
        f"필수 컬럼 누락: {missing_columns}"
    )

# 월별 컬럼 순서 통일
schemas = [
    pa.schema([schema.field(col) for col in columns])
    for schema in schemas
]

target_schema = pa.unify_schemas(
    schemas,
    promote_options="permissive",
)

# 모두 성공한 뒤 최종 파일명으로 변경
temp_path = output_path.with_suffix(".parquet.part")

total_rows = 0
expected_rows = 0
reports = []

with pq.ParquetWriter(
    temp_path,
    target_schema,
    compression="snappy",
) as writer:

    for i, (month, path) in enumerate(
        zip(months, paths),
        start=1,
    ):
        source = pq.ParquetFile(path)

        input_rows = source.metadata.num_rows
        expected_rows += input_rows

        written_rows = 0
        missing_labels = 0

        for batch in source.iter_batches(
            batch_size=BATCH_SIZE,
            use_threads=False,
        ):
            table = pa.Table.from_batches([batch])
            table = table.select(columns).cast(target_schema)

            # 기준월 확인 — 작은 배치의 해당 컬럼만 사용
            month_values = table["기준월"].to_pandas()

            if not month_values.eq(
                month.strftime("%Y-%m")
            ).fillna(False).all():
                raise ValueError(
                    f"{path.name}: 기준월이 일치하지 않습니다."
                )

            missing_labels += (
                table["앞으로6개월내_폐업여부"].null_count
            )

            writer.write_table(table)

            written_rows += table.num_rows
            total_rows += table.num_rows

            del month_values, table, batch

        if written_rows != input_rows:
            raise ValueError(
                f"{path.name}: 입력·출력 행 수 불일치"
            )

        reports.append({
            "기준월": month.strftime("%Y-%m"),
            "입력행수": input_rows,
            "저장행수": written_rows,
            "정답결측행수": missing_labels,
        })

        print(
            f"[{i:02d}/{len(paths)}] {month:%Y-%m} | "
            f"{written_rows:,}행 | "
            f"누적 {total_rows:,}행",
            flush=True,
        )

        del source
        gc.collect()

# 전체 행 수 검증
saved_rows = pq.ParquetFile(temp_path).metadata.num_rows

if not (saved_rows == total_rows == expected_rows):
    raise ValueError("전체 행 수 검증에 실패했습니다.")

temp_path.replace(output_path)

report = pd.DataFrame(reports)

report.to_csv(
    output_path.parent / "모델링용_전체_202301_202602_병합확인.csv",
    index=False,
    encoding="utf-8-sig",
)

print("\n저장 완료")
print("파일:", output_path)
print(f"전체 행 수: {saved_rows:,}")
print(f"컬럼 수: {len(target_schema.names):,}")
print(f"파일 크기: {output_path.stat().st_size / 1024**2:,.1f} MB")

display(report)

if report["정답결측행수"].sum() > 0:
    print(
        "정답이 결측인 행이 포함되어 있습니다. "
        "병합은 완료됐지만 실제 학습 전에 라벨 확인이 필요합니다."
    )

In [ ]:
source = pq.ParquetFile(output_path)

print(f"전체 행 수: {source.metadata.num_rows:,}")
print(f"전체 컬럼 수: {len(source.schema_arrow.names):,}")

industry_cols = [
    col for col in source.schema_arrow.names
    if col.startswith("업종__")
]

print(f"업종별 0·1 컬럼: {len(industry_cols)}개")

batch = next(
    source.iter_batches(
        batch_size=1_000,
        use_threads=False,
    ),
    None,
)

if batch is not None:
    df_model_sample = batch.to_pandas()
    display(df_model_sample.head(10))

del source, batch
gc.collect()

# 마지막 스냅샷이 포함됐는지 병합 요약으로 확인
display(report.tail(3))

In [ ]:
source = pq.ParquetFile(output_path)

print(f"전체 행 수: {source.metadata.num_rows:,}")
print(f"전체 컬럼 수: {len(source.schema_arrow.names):,}")

industry_cols = [
    col for col in source.schema_arrow.names
    if col.startswith("업종__")
]

print(f"업종별 0·1 컬럼: {len(industry_cols)}개")

batch = next(
    source.iter_batches(
        batch_size=1_000,
        use_threads=False,
    ),
    None,
)

if batch is not None:
    df_model_sample = batch.to_pandas()
    display(df_model_sample.head(10))

del source, batch
gc.collect()

# 마지막 스냅샷이 포함됐는지 병합 요약으로 확인
display(report.tail(3))